In [1]:
import pandas as pd


df = pd.read_csv('diamonds_full.csv',encoding='utf-8')

In [2]:
#ENCODING
from sklearn.preprocessing import OrdinalEncoder

cut_order    = ['Fair','Good','Very Good','Premium','Ideal']
color_order  = ['J','I','H','G','F','E','D']
clarity_order= ['I1','SI2','SI1','VS2','VS1','VVS2','VVS1','IF']

enc = OrdinalEncoder(categories=[cut_order, color_order, clarity_order], dtype=int)
df[['cut_ord','color_ord','clarity_ord']] = enc.fit_transform(df[['cut','color','clarity']])
df[['cut','cut_ord','color','color_ord','clarity','clarity_ord']].head()


,cut,cut_ord,color,color_ord,clarity,clarity_ord
0,Ideal,4,E,5,SI2,1
1,Premium,3,E,5,SI1,2
2,Good,1,E,5,VS1,4
3,Premium,3,I,1,VS2,3
4,Good,1,J,0,SI2,1


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle

features = ['carat','x','y','z','depth','table','volume','cut_ord','color_ord','clarity_ord']
X = df[features]
y = df['price_inr']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

# Save scaler in current directory
with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
!pip install xgboost


In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor

models = {
    'LinearRegression': LinearRegression(),
    'DecisionTree': DecisionTreeRegressor(random_state=42),
    'RandomForest': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost': XGBRegressor(n_estimators=100, random_state=42),
    'KNN': KNeighborsRegressor(),
    'ANN': MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=500, early_stopping=True, random_state=42)
}


In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

results = {}
best_model_name = None
best_r2 = -np.inf

for name, model in models.items():
    model.fit(X_train_s, y_train)
    preds = model.predict(X_test_s)

    mae = mean_absolute_error(y_test, preds)
    mse = mean_squared_error(y_test, preds)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, preds)

    results[name] = {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'R2': r2}
    print(f"{name} → MAE: {mae:.2f}, RMSE: {rmse:.2f}, R²: {r2:.4f}")

    if r2 > best_r2:
        best_r2 = r2
        best_model_name = name

with open(f'{best_model_name}.pkl', 'wb') as f:
    pickle.dump(models[best_model_name], f)



LinearRegression → MAE: 46494.18, RMSE: 66819.27, R²: 0.9118
DecisionTree → MAE: 22211.13, RMSE: 41262.20, R²: 0.9664
RandomForest → MAE: 16600.21, RMSE: 30545.71, R²: 0.9816
XGBoost → MAE: 17369.41, RMSE: 30601.65, R²: 0.9815
KNN → MAE: 22297.67, RMSE: 38609.56, R²: 0.9706
ANN → MAE: 28072.27, RMSE: 46239.13, R²: 0.9578
